# ADS v2: setup

Settings, the build window, the hash salt (the same as ods_v2) and functions, the run log. Run by every ads notebook through `%run ADS_v2_00_Setup`.


In [ ]:
%%pyspark
# ADS v2 settings. Every ads notebook runs this notebook first (%run ADS_v2_00_Setup).
# A value the calling notebook or pipeline has already set wins; anything not set takes the default here.
target_schema  = globals().get("target_schema")  or "ads_v2"
ods_schema     = globals().get("ods_schema")     or "ods_v2"
link_schema    = globals().get("link_schema")    or "ods"
pii_schema     = globals().get("pii_schema")     or "pii"
stg_schema     = globals().get("stg_schema")     or "stg"
start_time     = globals().get("start_time")     or "2021-07-01"      # first event time (inclusive)
end_time       = globals().get("end_time")       or "2026-09-01"      # last event time (exclusive)
effective_date = globals().get("effective_date") or "2026-08-31"      # month-end cut-off for warehouse facts
batch_id       = globals().get("batch_id")       or ""                # blank = generated from the start time
# The hash salt: the SAME value as the ods_v2 salt (ODS_v2_00_Setup), so ads hash keys match ods hash keys.
hash_salt     = "slkjdghwe234"


In [ ]:
%%pyspark
import datetime
if not hash_salt:
    raise ValueError("hash_salt is empty: set it in ADS_v2_00_Setup (the same value as the ods_v2 salt)")
batch_id = batch_id or datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%d%H%M%S")
conf = {"ads.target": target_schema, "ads.ods": ods_schema, "ads.link": link_schema, "ads.pii": pii_schema,
        "ads.stg": stg_schema, "ads.start_time": start_time, "ads.end_time": end_time,
        "ads.effective_date": effective_date, "ads.batch_id": batch_id}
spark.conf.set("spark.sql.variable.substitute", "true")
spark.conf.set("spark.sql.session.timeZone", "UTC")
for k, v in conf.items():
    spark.conf.set(k, v)
spark.sql(f"create schema if not exists {target_schema}")
print(conf)


# Hash functions used by the SQL cells. The salt stays in this closure: it is never written into SQL text or plans.
import hashlib, re
import pandas as pd
from pyspark.sql.functions import pandas_udf
_SALT = hash_salt
_EMAIL = re.compile(r"^[^@\s]+@[^@\s]+\.[^@\s]+$")
def _sha(v):
    return hashlib.sha256((_SALT + v).encode("utf-8")).hexdigest()
def _missing(x):                        # pandas hands an all-null batch over as NaN, not None
    return x is None or (isinstance(x, float) and x != x)
def _email(x):
    if _missing(x): return None
    v = str(x).strip().lower()
    return _sha(v) if _EMAIL.match(v) and v != "gdpr" else None
def _phone(x):
    if _missing(x): return None
    d = re.sub(r"[^0-9]", "", str(x))
    if len(d) < 7: return None
    if d.startswith("00353"): d = "0" + d[5:]
    elif d.startswith("353"): d = "0" + d[3:]
    elif len(d) == 9 and d.startswith("8"): d = "0" + d
    return _sha(d)
_ID_NULLS = {"", "GDPR", "NULL", "NONE", "NAN"}
def _id(x, extra):
    if _missing(x): return None
    v = re.sub(r"[^A-Za-z0-9]", "", str(x)).upper()
    return None if v in _ID_NULLS or (extra and v in extra.split(",")) else _sha(v)
_EMAIL_IN = re.compile(r"[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}")
_MOBILE_IN = re.compile(r"(?<![0-9A-Za-z-])(\+?353 ?|0)8[35-9][ -]?[0-9]{3}[ -]?[0-9]{4}(?![0-9A-Za-z-])")
def _redact(x):
    if _missing(x): return None
    return _MOBILE_IN.sub("[phone]", _EMAIL_IN.sub("[email]", str(x)))
def _area(x):
    if _missing(x): return None
    v = re.sub(r"[^A-Za-z0-9]", "", str(x)).upper()
    return v[:3] if len(v) >= 3 else None
@pandas_udf("string")
def ods_hash_email(s: pd.Series) -> pd.Series: return s.map(_email)
@pandas_udf("string")
def ods_hash_phone(s: pd.Series) -> pd.Series: return s.map(_phone)
@pandas_udf("string")
def ods_hash_id(s: pd.Series, extra: pd.Series) -> pd.Series: return pd.Series([_id(a, b) for a, b in zip(s, extra)])
@pandas_udf("string")
def ods_eircode_area(s: pd.Series) -> pd.Series: return s.map(_area)
@pandas_udf("string")
def ods_redact(s: pd.Series) -> pd.Series: return s.map(_redact)
for f in (ods_hash_email, ods_hash_phone, ods_hash_id, ods_eircode_area, ods_redact):
    spark.udf.register(f.__name__, f)

from pyspark.sql import functions as _F
def ads_run_log(notebook, status, detail=""):
    (spark.createDataFrame([(batch_id, notebook, status, detail)], "AdsBatchId string, Notebook string, Status string, Detail string")
          .withColumn("LoggedAt", _F.current_timestamp())
          .write.mode("append").saveAsTable(f"{target_schema}.AdsRunLog"))
